# Notebook 05 — Fan Segmentation

K-Means clustering on fan behavioral features. Selects k via silhouette score, assigns readable segment names, and profiles each cluster for business recommendations.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from src.config import DATA_PROCESSED_DIR, DATA_SIMULATED_DIR
sns.set_theme(style='whitegrid')
print('Setup complete')

## 1. Fan Feature Analysis

In [ ]:
for d in [DATA_PROCESSED_DIR, DATA_SIMULATED_DIR]:
    p = d / 'fan_profiles.csv'
    if p.exists(): df = pd.read_csv(p); break

features = ['games_attended_last_12_months','avg_ticket_spend','avg_concession_spend',
            'avg_merchandise_spend','promotion_usage_rate','email_engagement_score','fan_value_score']

print(f'Fan profiles: {len(df):,}')
display(df[features].describe().round(2))

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for i, feat in enumerate(features):
    ax = axes[i // 4][i % 4]
    ax.hist(df[feat], bins=30, color='steelblue', edgecolor='white')
    ax.set_title(feat)
axes[1][3].set_visible(False)
plt.suptitle('Fan Feature Distributions')
plt.tight_layout()
plt.show()

## 2. K-Means Clustering — Silhouette Score Selection

In [ ]:
X = df[features].fillna(0).values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

scores = {}
for k in range(3, 9):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    scores[k] = silhouette_score(X_scaled, labels, sample_size=5000)

plt.figure(figsize=(8, 4))
plt.plot(list(scores.keys()), list(scores.values()), marker='o', color='steelblue')
plt.xlabel('k (number of clusters)')
plt.ylabel('Silhouette Score')
plt.title('Silhouette Score by Number of Clusters')
plt.tight_layout()
plt.show()
best_k = max(scores, key=scores.get)
print(f'Optimal k: {best_k} (silhouette = {scores[best_k]:.4f})')

## 3. Train Final Segmentation Model

In [ ]:
from src.segmentation import main as run_segmentation
run_segmentation()

## 4. Segment Profile Analysis

In [ ]:
seg_df = pd.read_csv(DATA_PROCESSED_DIR / 'fan_profiles_segmented.csv')
seg_col = 'fan_segment' if 'fan_segment' in seg_df.columns else 'loyalty_status'

profile = seg_df.groupby(seg_col)[features + ['games_attended_last_12_months']].mean().round(2)
display(profile)

# Spider / radar chart per segment
profile_norm = (profile - profile.min()) / (profile.max() - profile.min())
fig, ax = plt.subplots(figsize=(12, 5))
profile_norm[['avg_ticket_spend','avg_concession_spend','avg_merchandise_spend',
              'promotion_usage_rate','email_engagement_score','games_attended_last_12_months']].T.plot(
    kind='bar', ax=ax)
ax.set_title('Normalized Segment Feature Profiles')
ax.set_ylabel('Normalized Score')
plt.xticks(rotation=25, ha='right')
plt.tight_layout()
plt.show()

## 5. Business Recommendations

- **High-Value Fans / Season Ticket Loyalists:** Retention-first. Offer exclusive early access, personalized outreach, premium renewal packages.
- **Promo-Sensitive Fans:** Target with time-limited promotions on low-demand games. Do not discount premium inventory.
- **Family Buyers:** Maximize with weekend promotions, family packs, combo meal deals. High concession/merch spend.
- **Merchandise-Heavy Fans:** New jersey launches, rivalry game specials, and limited edition items drive this segment.
- **Casual Low-Frequency Fans:** Re-engagement campaigns. Focus on memorable first-visit experience to drive frequency.
- **Corporate / Group Guests:** Streamline group sales process. Bundle experience packages with corporate partner programs.